<a href="https://colab.research.google.com/github/joel-shor/AEAG/blob/master/notebooks/DNA_BERT_S_golden_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A quick test to generate some pairs of "golden tests", of simple (input, output) pairs, to compare the correctness of more complicated workflows.

All code is based on the "quick start" in the official github repo [here](https://github.com/MAGICS-LAB/DNABERT_S?tab=readme-ov-file#4-quick-start)

**NOTE**: This notebook must be run with a GPU runtime. GPUs other than A100 might have trouble with triton.

In [1]:
# If not on an A100, run the next line.
!pip uninstall -y triton

Found existing installation: triton 3.2.0
Uninstalling triton-3.2.0:
  Successfully uninstalled triton-3.2.0


In [2]:
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-S", trust_remote_code=True)
model = AutoModel.from_pretrained("zhihan1996/DNABERT-S", trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/268 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/168k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

... (5 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNABERT-S:
- flash_attn_triton.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNABERT-S:
- bert_layers.py
- bert_padding.py
- flash_attn_triton.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/468M [00:00<?, ?B/s]

/root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-S/2efd650282ec5d5ab377c787c76ea56b723f6b7c/bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(


model.safetensors:   0%|          | 0.00/468M [00:00<?, ?B/s]

In [3]:
import numpy as np

# This is (slightly modified) code from the repo. We try it once, then generalize in the next
# cell.
def mean_pool_embedding(dna: str) -> np.ndarray:
  inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"]
  hidden_states = model(inputs)[0] # [1, sequence_length, 768]

  # embedding with mean pooling
  embedding_mean = torch.mean(hidden_states[0], dim=0)
  assert embedding_mean.shape == (768,)

  return embedding_mean.detach().numpy()

dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
mean_pool_embedding(dna)

array([ 3.32480483e-02,  1.25162527e-01, -1.26686037e-01, -2.00645551e-01,
        5.40220318e-03,  1.09037980e-02,  6.53505772e-02,  6.66581243e-02,
       -3.41409035e-02,  4.66810502e-02, -1.31082132e-01,  1.95729300e-01,
        5.13276979e-02, -1.07978784e-01, -6.90421239e-02, -1.46670742e-02,
       -3.20461281e-02, -7.14279488e-02,  1.42731825e-02,  5.27767837e-02,
        1.43181697e-01,  3.35092813e-01,  5.04306853e-02,  5.78454472e-02,
        1.72885507e-01,  2.68949475e-02,  3.42365235e-01, -1.15089931e-01,
       -8.32027942e-02, -3.00971754e-02, -3.22929509e-02,  1.62377745e-01,
       -4.41136360e-01, -1.20534450e-01, -2.01645657e-01, -5.43290526e-02,
        1.39968023e-02, -1.07276849e-01,  2.24474803e-01, -2.53390186e-02,
       -1.94194928e-01,  4.45796475e-02, -6.03195932e-03,  1.55062294e-02,
       -1.31078392e-01,  1.60389483e-01, -4.90302965e-02,  4.33664024e-02,
        1.55607775e-01,  2.28951231e-01,  6.50050342e-02, -1.37010738e-01,
        9.99399740e-03, -

In [4]:
dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
import random
def _random_dna(seq_len=len(dna)):
  return ''.join([random.choice('AGTC') for _ in range(seq_len)])

dnas = [dna] + [_random_dna() for _ in range(4)]

golden_input_output_pairs = []
for input_dna in dnas:
  output = mean_pool_embedding(input_dna)
  golden_input_output_pairs.append((input_dna, output))

import pickle
with open('golden_input_output_pairs.pkl', 'wb') as f:
  pickle.dump(golden_input_output_pairs, f)

In [5]:
# Sample code to show how to load it.
import pickle
with open('golden_input_output_pairs.pkl', 'rb') as f:
  golden_input_output_pairs = pickle.load(f)
for input_dna, output in golden_input_output_pairs:
  print(input_dna, output)

ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC [ 3.32480483e-02  1.25162527e-01 -1.26686037e-01 -2.00645551e-01
  5.40220318e-03  1.09037980e-02  6.53505772e-02  6.66581243e-02
 -3.41409035e-02  4.66810502e-02 -1.31082132e-01  1.95729300e-01
  5.13276979e-02 -1.07978784e-01 -6.90421239e-02 -1.46670742e-02
 -3.20461281e-02 -7.14279488e-02  1.42731825e-02  5.27767837e-02
  1.43181697e-01  3.35092813e-01  5.04306853e-02  5.78454472e-02
  1.72885507e-01  2.68949475e-02  3.42365235e-01 -1.15089931e-01
 -8.32027942e-02 -3.00971754e-02 -3.22929509e-02  1.62377745e-01
 -4.41136360e-01 -1.20534450e-01 -2.01645657e-01 -5.43290526e-02
  1.39968023e-02 -1.07276849e-01  2.24474803e-01 -2.53390186e-02
 -1.94194928e-01  4.45796475e-02 -6.03195932e-03  1.55062294e-02
 -1.31078392e-01  1.60389483e-01 -4.90302965e-02  4.33664024e-02
  1.55607775e-01  2.28951231e-01  6.50050342e-02 -1.37010738e-01
  9.99399740e-03 -1.94825172e-01  3.44012007e-02  8.88273194e-02
 -1.00565024e-01 -9.23571363e-